# 평가 분석 20분 — 같은 자로 세 번 재고, 돈으로 바꾼다

6회차 수업에서 씁니다. 셀 번호는 코드 셀 첫 줄의 `# [셀 N]` 표시입니다. 이 노트북은 Qdrant 를 열지 않습니다(셀 3 만 LLM 을 한 번 부릅니다). 터미널에서 `make judge` · `make calc-e2e` 가 도는 동안 같이 써도 됩니다.

In [ ]:
# [셀 1] 결과 파일 셋을 읽는다 — 같은 40문항, 같은 gold, 다른 파이프라인
%load_ext autoreload
%autoreload 2   # 정답을 복사한 뒤 셀을 다시 실행하면 새 코드를 본다. 커널 재시작 불필요
import sys, json
from pathlib import Path
sys.path.insert(0, "../src")
R = Path("../results")
res = {n: json.loads((R / f"{n}.json").read_text(encoding="utf-8")) for n in ("baseline", "hybrid", "agentic") if (R / f"{n}.json").exists()}
print("읽은 파일:", list(res))
for n, r in res.items():
    o = r["overall"]
    print(f"{n:9} n={r['n']}  만든 날 {r['generated_at'][:10]}  llm={r.get('llm')}  "
          f"R@5 {o['recall@5']:.1%}  MRR {o['mrr']:.3f}  " + (f"답변률 {o.get('answer_rate'):.1%}" if o.get("answer_rate") is not None else ""))
# 행 하나가 문항 하나다. 세 파일의 행 수와 문항 ID 가 같아야 한 표에 놓을 수 있다
ids = {n: [row["id"] for row in r["rows"]] for n, r in res.items()}
print("같은 문항 순서:", all(v == list(ids.values())[0] for v in ids.values()))
print("agentic 행 하나의 키:", sorted(res["agentic"]["rows"][0].keys()))

## 1. 골든셋 검수 — 유형 8개, 거절해야 할 문항 2개

`data/golden/seed_golden.jsonl` 의 40문항입니다. 3회차부터 한 번도 바꾸지 않았습니다. 바꾸면 세 파일을 다시 만들어야 해서 비교가 깨집니다. `answerable=false` 인 둘(S39 · S40)이 "거절해야 할 문항"이고, 미답변 정확도의 분모입니다.

In [ ]:
# [셀 2] 골든셋 40문항 — 유형별 수, 거절해야 할 문항, gold 청크 ID
from collections import Counter
from finrag.eval import harness
golden, gold_ids = harness.load_golden()
print("문항", len(golden), "유형별", dict(Counter(g["type"] for g in golden)))
print("거절해야 할 문항:", [g["id"] for g in golden if not g.get("answerable", True)])
print("기대 필터가 적힌 문항:", sum(1 for g in golden if g.get("expected_filters")), "개")
g = golden[0]
print(f"\n{g['id']} {g['type']}  {g['question'][:50]}…")
print("  gold 청크:", gold_ids[g["id"]])
print("  정답 요지:", g["answer"][:60], "…")
# 지표 다섯의 분모. 여기가 실습 1 의 핵심이다
rows = res["agentic"]["rows"]
can = [r for r in rows if r.get("answerable", True)]; cannot = [r for r in rows if not r.get("answerable", True)]
print(f"\n답할 수 있는 문항 {len(can)} (오거절률의 분모) · 거절해야 할 문항 {len(cannot)} (미답변 정확도의 분모)")

## 2. LLM-as-Judge — 루브릭 셋, 0·1·2

자유 서술로 점수를 매기게 하면 같은 답변에 다른 점수가 나옵니다. 항목과 척도를 고정하고 한 줄 이유를 요구합니다. 아래 셀이 `RUBRIC` 을 찍고, 20문항 중 하나를 실제로 채점합니다(키가 없으면 근거 일치만 기계 채점).

In [ ]:
# [셀 3] 루브릭과 실제 채점 한 문항 (LLM 호출 1회)
from finrag.eval import judge
print(judge.RUBRIC)
qid = "S01"
item = next(g for g in golden if g["id"] == qid)
row = next(r for r in res["agentic"]["rows"] if r["id"] == qid)
cited = row.get("chunk_ids") or [row.get("top_chunk")]
sc = judge.score_one(item["question"], row["answer"] or "", item["answer"], gold_ids[qid], cited)
print(f"\n{qid} 답변 앞부분: {(row['answer'] or '')[:120]}…")
print(f"인용한 청크: {cited[:3]}   gold: {gold_ids[qid]}")
print(f"judge → 근거 {sc.basis_match} · 수치 {sc.numeric_ok} · 근거 밖 주장 없음 {sc.no_extra_claim} = {sc.total}/6   이유: {sc.reason}")

## 3. 일치도 — judge 를 믿기 전에 (실습 2)

같은 답변 20개를 사람(강사)과 judge 가 각각 채점한 두 열이 있으면 일치도가 나옵니다. 완전 일치 · ±1 이내 · 평균 절대 오차(MAE). 아래는 실습 2 의 `agreement` 를 장난감 숫자로 돌린 것입니다. 실습 2 를 채우기 전에는 `NotImplementedError` 입니다. 실제 20문항은 터미널의 `make judge-calibrate` 가 합니다.

In [ ]:
# [셀 4] agreement 장난감 (실습 2 뒤에 다시 실행)
judge_scores = [6, 5, 4, 2, 0, 6, 3]
human_labels = [6, 4, 4, 4, 1, 5, 3]
try:
    a = judge.agreement(judge_scores, human_labels)
    print("완전 일치", a["exact"], "· ±1 이내", a["within_1"], "· MAE", a["mae"], "· n", a["n"])
    bias = sum(j - h for j, h in zip(judge_scores, human_labels)) / len(judge_scores)
    print(f"judge − 사람 평균 {bias:+.2f}  → 양수면 judge 가 후하다")
except NotImplementedError as e:
    print("아직 안 채움:", e)

## 4. 비용 — 토큰을 돈으로

5회차 결과 행에 `usage`(모델별 입력·출력 토큰)가 있습니다. 검색은 로컬이라 0 이고 돈은 LLM 호출에만 듭니다. 단가는 `eval/cost.py` 의 표(2026-10-08 기준)입니다. 바뀌면 그 표만 고칩니다.

In [ ]:
# [셀 5] 문항당 토큰과 비용, 가장 비싼 문항
from finrag.eval import cost
s = cost.summarize(res["agentic"])
print(f"usage 가 있는 문항 {s['n_with_usage']}/{s['n']}   문항당 입력 {s['tokens_in_avg']:,} · 출력 {s['tokens_out_avg']:,} 토큰   "
      f"문항당 ${s['cost_usd_avg']:.5f} = {s['cost_krw_avg']:.1f}원   재검색한 문항 {s['retry_share']:.0%}")
for m, b in s["by_model"].items():
    print(f"  {m:28} 입력 {b['input_avg']:>6,}  출력 {b['output_avg']:>5,}")
priced = sorted(((cost.cost_usd(r.get("usage")), r["id"], r.get("retry_count", 0), len(r.get("answer") or "")) for r in res["agentic"]["rows"] if r.get("usage")), reverse=True)
print("\n비싼 문항 셋 (비용, 재검색 횟수, 답 길이):")
for c, qid, rc, ln in priced[:3]:
    print(f"  {qid}  ${c:.4f}  재검색 {rc}회  답 {ln:,}자")
m = cost.monthly(res["agentic"], daily_questions=500)
print(f"\n일 500건 × 22일이면 월 {m['monthly_krw']:,}원 (환율 {m['usd_krw']:,.0f} 가정)")

## 5. 미답변 정확도 vs 오거절률 — 2×2 표

오거절률의 분모는 "답할 수 있는 38", 미답변 정확도의 분모는 "거절해야 할 2" 입니다. 한 칸을 줄이면 다른 칸이 늡니다.

In [ ]:
# [셀 6] 2×2 — 답할 수 있나 × 답했나
rows = res["agentic"]["rows"]
def cell(answerable, answered): return [r["id"] for r in rows if r.get("answerable", True) == answerable and bool(r.get("answered")) == answered]
a_a, a_r = cell(True, True), cell(True, False)
c_a, c_r = cell(False, True), cell(False, False)
limited = [r["id"] for r in rows if not r.get("answerable", True) and r.get("answered") and r.get("grade") == "insufficient"]
print(f"{'':16}{'답했다':>8}{'거절했다':>10}")
print(f"{'답할 수 있는':16}{len(a_a):>8}{len(a_r):>10}   ← 오거절 {a_r}")
print(f"{'거절해야 할':16}{len(c_a):>8}{len(c_r):>10}   ← 답한 것 중 한계를 밝힘 {limited}")
print(f"\n오거절률 {len(a_r)}/{len(a_a)+len(a_r)} = {len(a_r)/(len(a_a)+len(a_r)):.1%}   미답변 정확도 {len(limited)+len(c_r)}/{len(c_a)+len(c_r)} = {(len(limited)+len(c_r))/(len(c_a)+len(c_r)):.0%}")
print("거절한 문항의 판정:", {r["id"]: r.get("grade") for r in rows if not r.get("answered")})

## 6. 필터 정확도 — 틀리면 조용히 0건

골든셋의 `expected_filters` 와 에이전트가 뽑은 `filters` 를 필드별로 맞춰 봅니다. 터미널의 `make filter-accuracy` 와 같은 함수입니다.

In [ ]:
# [셀 7] 필터 정확도 — 필드별 정밀도·재현율과 어긋난 문항
from finrag.eval.filters_eval import FIELDS, filter_accuracy
fa = filter_accuracy(res["agentic"]["rows"], golden)
print(f"기대 필터가 적힌 {fa['n_expected']}문항 중 기대한 필드가 전부 맞은 것 {fa['exact_match']}   (기대 없는 {fa['n_no_expected']}문항은 평가하지 않음)")
for k in FIELDS:
    c = fa["by_field"][k]
    if c["n"]:
        print(f"  {k:13} 기대 {c['n']:>2}  맞음 {c['tp']:>2}  다른 값 {c['wrong']:>2}  놓침 {c['fn']:>2}   정밀도 {c['precision'] if c['precision'] is not None else '-'}  재현율 {c['recall'] if c['recall'] is not None else '-'}")
for m in fa["mismatches"][:8]:
    print(f"  {m['id']} {m['field']:12} 기대 {m['expected']!s:14} 뽑음 {m['got']}")

## 7. 오늘 코드에서는 어디에 있나

| 이 노트북 | 오늘 코드 |
|---|---|
| 셀 1 의 결과 파일 읽기 | `src/finrag/eval/report.py` 의 `load` — 파일 요약을 믿지 않고 `rows` 를 다시 집계(**실습 1** `metrics.aggregate`) |
| 셀 2 의 분모 둘 | `metrics.aggregate` 의 `false_refusal_rate` · `abstain_accuracy` |
| 셀 3 의 루브릭 | `src/finrag/eval/judge.py` 의 `RUBRIC` · `score_one`. 20문항은 `scripts/judge_run.py` |
| 셀 4 의 일치도 | `judge.agreement` (**실습 2**), `judge.calibrate` |
| 셀 5 의 비용 | `src/finrag/eval/cost.py`, `scripts/roi.py` |
| 셀 6 의 2×2 | `metrics.aggregate` 가 같은 것을 비율로 |
| 셀 7 의 필터 정확도 | `src/finrag/eval/filters_eval.py`, `scripts/filter_accuracy.py` |